In [1]:
# Cell 1 — Imports and setup
import os, math, textwrap, random
import numpy as np
import pandas as pd
from pathlib import Path

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, f1_score
from sklearn.model_selection import train_test_split

torch.__version__, torch.cuda.is_available()


('2.5.1', True)

In [2]:
# Cell 2 — Reproducibility helpers
def set_seed(seed: int = 123):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Determinism trade-offs
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(863)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE


'cuda'

In [4]:
# Cell 3 — Create or load dataset
# Expect a CSV with 9 normalized features and a binary label in the last column named 'label'.
# If not found, we synthesize a well-behaved toy dataset.

DATA_DIR = Path("05 Intermediate Deep Learning with Pytorch/artifacts/data")
DATA_DIR.mkdir(exist_ok=True)
CSV_PATH = DATA_DIR / "water_train.csv"

if not CSV_PATH.exists():
    # Synthesize 10_000 samples, 9 features in [0,1], with a nonlinear label
    rng = np.random.default_rng(42)
    N = 10_000
    X = rng.uniform(0, 1, size=(N, 9))
    # Nonlinear boundary + noise; control class balance with threshold
    s = (
        1.5*X[:,0] - 1.2*X[:,1] + 0.8*X[:,2]**2
        - 0.9*np.sin(2*np.pi*X[:,3])
        + 0.6*X[:,4]*X[:,5]
        - 0.7*(X[:,6]-0.5)**2
        + 0.5*X[:,7] - 0.4*X[:,8]
    )
    s = (s - s.mean())/s.std()
    p = 1/(1+np.exp(-s))
    y = (rng.uniform(0,1,size=N) < p).astype(int)
    df = pd.DataFrame(X, columns=[f"feat_{i}" for i in range(9)])
    df["label"] = y
    df.to_csv(CSV_PATH, index=False)

# Train/val split persisted to disk for reproducibility
df = pd.read_csv(CSV_PATH)
train_df, val_df = train_test_split(df, test_size=0.2, random_state=863, stratify=df["label"])
train_df.to_csv(DATA_DIR/"water_train_split.csv", index=False)
val_df.to_csv(DATA_DIR/"water_val_split.csv", index=False)

train_df.head(), train_df["label"].value_counts(normalize=True).round(3)


(        feat_0    feat_1    feat_2    feat_3    feat_4    feat_5    feat_6  \
 7652  0.479362  0.966016  0.667444  0.132326  0.212284  0.655472  0.947787   
 9442  0.352440  0.973784  0.695551  0.521487  0.446856  0.684043  0.893005   
 7172  0.971946  0.923282  0.145343  0.789623  0.212462  0.257353  0.491019   
 9784  0.933560  0.639928  0.506173  0.125420  0.889409  0.600508  0.137574   
 5115  0.849384  0.815674  0.785437  0.780377  0.944451  0.605411  0.827936   
 
         feat_7    feat_8  label  
 7652  0.332398  0.116993      1  
 9442  0.230543  0.741483      0  
 7172  0.802976  0.443161      1  
 9784  0.482303  0.037066      0  
 5115  0.801341  0.270234      0  ,
 label
 1    0.501
 0    0.499
 Name: proportion, dtype: float64)